In [13]:
%%capture --no-stderr
%pip install ragas datasets tqdm import_ipynb llama-index-embeddings-huggingface llama-index-llms-mistralai llama-index-llms-anthropic langchain-huggingface




In [17]:
# Core imports
import os
import sys
import json
import importlib
import re
import string
import traceback
import copy
import math
import collections
from typing import Any, List, Mapping, Optional, Dict
import types

# Data processing
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import Dataset

# Environment
from dotenv import load_dotenv
import import_ipynb

# LlamaIndex
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    Document
)
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.mistralai import MistralAI
from llama_index.llms.openai import OpenAI
from llama_index.llms.anthropic import Anthropic
from langchain_huggingface import HuggingFaceEmbeddings

from mistralai import Mistral
from sentence_transformers import CrossEncoder

# RAGAS evaluation
from ragas import evaluate, EvaluationDataset
from ragas.llms import LlamaIndexLLMWrapper
from ragas.metrics import (
    Faithfulness, 
    FactualCorrectness,
    ResponseRelevancy,

    ResponseGroundedness,
    ContextRelevance,
    AnswerAccuracy,
    ContextPrecision,
    ContextRecall
)

# Local imports
from Sustainability_expert import SustainabilityExpert


In [18]:
import sys 
import importlib

if 'Sustainability_expert' in sys.modules: # sys provides access to variables and functions in the python interpreter. 'sys.modules contains all the currently loaded modules in Python.

    importlib.reload(sys.modules['Sustainability_expert'])   # forces Python to reload a previously imported module from disk

    # Now we can import the refreshed module

    from Sustainability_expert import SustainabilityExpert


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# CAN EDIT

In [21]:
load_dotenv()

class ExcelRAGEvaluator:
    
    def __init__(self, original_rag_system, excel_file="squad_2.0_for_eval.xlsx", force_rebuild=False):
        """
        Initialize the evaluator with original RAG system and Excel file
        
        Args:
            original_rag_system: Your existing RAG system (SustainabilityExpert)
            excel_file: Path to the Excel file with questions, answers, contexts
            force_rebuild: Whether to force rebuild the vector DB
        """
        # Store the original RAG system 
        self.original_rag = original_rag_system
        
        # Load the Excel dataset
        self.excel_file = excel_file
        self.dataset = self._load_excel_data()
        
        # Set up vector DB path
        self.index_dir = f"./excel_eval_index_{os.path.basename(excel_file).replace('.xlsx', '')}"
        
        # Force rebuild if requested
        if force_rebuild and os.path.exists(self.index_dir):
            print(f"🔄 Removing existing vector DB at {self.index_dir}")
            shutil.rmtree(self.index_dir)
        
        # Initialize BOTH embedding models (singular for LlamaIndex, plural for LangChain/RAGAS)
        self.embed_model = HuggingFaceEmbedding(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            device="cpu"
        )
        
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
        # Create the vector database from Excel contexts
        self._build_vector_db()
        
        # Create a modified RAG system that uses the Excel contexts
        self.evaluation_rag = self._create_evaluation_rag()
        
        self.llm = OpenAI(
            model="gpt-3.5-turbo",
            temperature=0,
            api_key=os.environ.get("OPENAI_API_KEY")
        )

        # Initialize the LLM wrapper for RAGAS
        self.eval_llm = LlamaIndexLLMWrapper(self.llm)
        
        # Set up RAGAS metrics - including all the metrics we want to evaluate
        self.metrics = [
            Faithfulness(),
            ContextPrecision(),
            ContextRecall(),
            ResponseRelevancy(),
            AnswerAccuracy(),
            ContextRelevance(),
            ResponseGroundedness(), 
        ]
    
    def _parse_string_list(self, string_list):
        """Parse a string representation of a list safely"""
        if not string_list or not isinstance(string_list, str):
            return []
            
        # Handle empty list
        cleaned = string_list.strip()
        if cleaned.startswith('[') and cleaned.endswith(']'):
            cleaned = cleaned[1:-1]
        if not cleaned:
            return []
            
        # Try using ast.literal_eval for safety, then fall back to manual parsing
        try:
            return ast.literal_eval(f"[{cleaned}]")
        except:
            items = []
            for item in cleaned.split(','):
                item = item.strip()
                if item.startswith("'") and item.endswith("'"):
                    item = item[1:-1]
                elif item.startswith('"') and item.endswith('"'):
                    item = item[1:-1]
                if item:
                    items.append(item)
            return items
    
    def _load_excel_data(self):
        """Load and parse the dataset from Excel file"""
        try:
            # Read the Excel file
            df = pd.read_excel(self.excel_file)
            
            print(f"✅ Loaded dataset from {self.excel_file}")
            print(f"📊 Dataset shape: {df.shape}")
            return df
            
        except Exception as e:
            print(f"❌ Error loading Excel dataset: {str(e)}")
            return pd.DataFrame()
    
    def _build_vector_db(self):
        """Build a vector DB from unique contexts in the Excel file"""
        # Skip if already built
        if os.path.exists(self.index_dir) and os.path.isdir(self.index_dir):
            print(f"✅ Vector DB already exists at {self.index_dir}")
            try:
                # Load existing index
                vector_store = FaissVectorStore.from_persist_dir(self.index_dir)
                storage_context = StorageContext.from_defaults(
                    vector_store=vector_store,
                    persist_dir=self.index_dir
                )
                self.index = load_index_from_storage(
                    storage_context=storage_context,
                    embed_model=self.embed_model
                )
                return
            except Exception as e:
                print(f"❌ Error loading existing index: {str(e)}")
                print("Will rebuild index...")
        
        if self.dataset.empty:
            print("❌ No valid data found")
            return
        
        print("🔄 Building vector DB from unique contexts...")
        
        # Extract all unique contexts
        contexts = []
        context_set = set()
        
        # Check if context column exists
        if 'context' not in self.dataset.columns:
            print(f"❌ Context column 'context' not found in Excel file")
            return
        
        # Extract contexts and create Document objects
        for idx, row in tqdm(self.dataset.iterrows(), total=len(self.dataset), desc="Processing contexts"):
            context = str(row['context']).strip() if pd.notna(row['context']) else ""
            
            # Skip if empty context or already seen
            if not context or context in context_set:
                continue
            
            # Add to our set of unique contexts
            context_set.add(context)
            
            # Create a Document object
            doc = Document(
                text=context,
                metadata={
                    "source": f"excel_{idx}",
                    "title": str(row.get('title', f"Document_{idx}"))
                }
            )
            contexts.append(doc)
        
        print(f"📚 Created corpus with {len(contexts)} unique paragraphs")
        
        # Create vector store
        vector_store = FaissVectorStore(dim=self.embed_model.embed_dim)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)
        
        # Create the index
        self.index = VectorStoreIndex(
            contexts,
            storage_context=storage_context,
            embed_model=self.embed_model
        )
        
        # Save index to disk
        os.makedirs(self.index_dir, exist_ok=True)
        storage_context.persist(persist_dir=self.index_dir)
        
        print(f"✅ FAISS index built and saved to {self.index_dir}")
    
    def _create_evaluation_rag(self):
        """Create a copy of the original RAG system with Excel-based vector store"""
        # Make a shallow copy of the original RAG system
        eval_rag = copy.copy(self.original_rag)
        
        try:
            # Print which model is being evaluated - this is the only print we'll keep
            if hasattr(self.original_rag, 'model'):
                print(f"Evaluating model: {self.original_rag.model}")
                # Ensure model is copied to evaluation RAG
                eval_rag.model = self.original_rag.model
                
            # Copy client for API-based models
            if hasattr(self.original_rag, 'client'):
                eval_rag.client = self.original_rag.client
                
            # Copy streaming flag if present
            if hasattr(self.original_rag, 'use_streaming'):
                eval_rag.use_streaming = False  # Always disable streaming for evaluation
            
            # Copy the LLM object if present
            if hasattr(self.original_rag, 'llm'):
                eval_rag.llm = self.original_rag.llm
                
            # Copy prompt templates - "original prompt template" refers to the template
            # used to format context and question before sending to the LLM
            if hasattr(self.original_rag, 'qa_prompt'):
                eval_rag.qa_prompt = self.original_rag.qa_prompt
            elif hasattr(self.original_rag, 'qa_prompt_template'):
                eval_rag.qa_prompt_template = self.original_rag.qa_prompt_template
            
            # Set up the retriever with Excel-based index
            eval_rag.index = self.index
            
            # Use same retriever settings
            retriever_kwargs = {}
            if hasattr(self.original_rag.retriever, 'similarity_top_k'):
                retriever_kwargs['similarity_top_k'] = self.original_rag.retriever.similarity_top_k
                print(f"Using same similarity_top_k={retriever_kwargs['similarity_top_k']} as original RAG")
            
            # Replace retriever with our index's retriever
            eval_rag.retriever = self.index.as_retriever(**retriever_kwargs)
            
            # Keep the reranker
            if hasattr(self.original_rag, 're_ranker'):
                eval_rag.re_ranker = self.original_rag.re_ranker
            
            # Define rerank_documents method
            def rerank_documents(self, query, nodes):
                """
                Re-rank nodes using cross-encoder
                Handles NodeWithScore objects properly
                """
                # Safety check - don't try to rerank if no nodes
                if not nodes:
                    return []
                
                # Extract text and preserve original nodes
                node_texts = []
                original_nodes = []
                
                for node in nodes:
                    try:
                        # Try different approaches to get text based on node type
                        if hasattr(node, 'node') and hasattr(node.node, 'text'):
                            # NodeWithScore case
                            text = node.node.text
                            original_nodes.append(node)
                            node_texts.append(text)
                        elif hasattr(node, 'text'):
                            # Direct TextNode case
                            text = node.text
                            original_nodes.append(node)
                            node_texts.append(text)
                        elif hasattr(node, 'get_content'):
                            # Other node types
                            text = node.get_content()
                            original_nodes.append(node)
                            node_texts.append(text)
                        elif hasattr(node, 'source_node') and hasattr(node.source_node, 'get_content'):
                            text = node.source_node.get_content()
                            original_nodes.append(node)
                            node_texts.append(text)
                        else:
                            # Try converting node to string as a last resort
                            text = str(node)
                            original_nodes.append(node)
                            node_texts.append(text)
                    except Exception as e:
                        print(f"Skipping node in reranking due to error: {str(e)}")
                        continue
                
                # Skip reranking if no valid nodes
                if not node_texts:
                    return nodes[:3] if len(nodes) > 3 else nodes
                
                try:
                    # Use the original reranker
                    reranker = self.re_ranker if hasattr(self, 're_ranker') else self.reranker
                    scores = reranker.predict([(query, text) for text in node_texts])
                    
                    # Sort by score and return top 4 nodes (to match original RAG using 4)
                    scored_pairs = sorted(zip(original_nodes, scores), key=lambda x: x[1], reverse=True)
                    top_nodes = [node for node, _ in scored_pairs[:4]]
                    
                    print(f"✅ Reranked to top {len(top_nodes)} documents")
                    return top_nodes
                    
                except Exception as e:
                    print(f"⚠️ Error during reranking: {str(e)}")
                    # Return original nodes if reranking fails
                    return nodes[:3] if len(nodes) > 3 else nodes
            
            # Attach the rerank_documents method to eval_rag
            eval_rag.rerank_documents = types.MethodType(rerank_documents, eval_rag)
            
            # Define safe_query method for evaluation
            def safe_query(self, question):
                """Safely query the RAG system with error handling"""
                try:
                    # Get nodes from retriever
                    nodes = self.retriever.retrieve(question)
                    
                    if not nodes:
                        return {
                            "answer": "No relevant information found.",
                            "contexts": []
                        }
                    
                    # Extract contexts
                    context_texts = []
                    
                    # Rerank nodes if available
                    if hasattr(self, 'rerank_documents'):
                        try:
                            reranked_nodes = self.rerank_documents(question, nodes)
                            
                            # Extract text from reranked nodes
                            for node in reranked_nodes:
                                try:
                                    if hasattr(node, 'node') and hasattr(node.node, 'text'):
                                        context_texts.append(node.node.text)
                                    elif hasattr(node, 'text'):
                                        context_texts.append(node.text)
                                    elif hasattr(node, 'get_content'):
                                        context_texts.append(node.get_content())
                                    elif hasattr(node, 'source_node') and hasattr(node.source_node, 'get_content'):
                                        context_texts.append(node.source_node.get_content())
                                    else:
                                        # Try converting to string as a last resort
                                        context_texts.append(str(node))
                                except Exception as e:
                                    print(f"Error extracting text from node: {str(e)}")
                        except Exception as e:
                            # Fall back to original nodes
                            for node in nodes:
                                try:
                                    if hasattr(node, 'node') and hasattr(node.node, 'text'):
                                        context_texts.append(node.node.text)
                                    elif hasattr(node, 'text'):
                                        context_texts.append(node.text)
                                    elif hasattr(node, 'get_content'):
                                        context_texts.append(node.get_content())
                                    elif hasattr(node, 'source_node') and hasattr(node.source_node, 'get_content'):
                                        context_texts.append(node.source_node.get_content())
                                    else:
                                        context_texts.append(str(node))
                                except Exception as e:
                                    print(f"Error extracting text from node: {str(e)}")
                    else:
                        # No reranking, extract text from original nodes
                        for node in nodes:
                            try:
                                if hasattr(node, 'node') and hasattr(node.node, 'text'):
                                    context_texts.append(node.node.text)
                                elif hasattr(node, 'text'):
                                    context_texts.append(node.text)
                                elif hasattr(node, 'get_content'):
                                    context_texts.append(node.get_content())
                                elif hasattr(node, 'source_node') and hasattr(node.source_node, 'get_content'):
                                    context_texts.append(node.source_node.get_content())
                                else:
                                    context_texts.append(str(node))
                            except Exception as e:
                                print(f"Error extracting text from node: {str(e)}")
                    
                    # Build context
                    context = "\n\n".join(context_texts) if context_texts else "No context available."
                    
                    # Generate answer based on the client type
                    try:
                        if hasattr(self, 'client') and self.client and hasattr(self, 'model'):
                            # Format the prompt with empty user_data for evaluation
                            if hasattr(self, 'qa_prompt_template'):
                                # Check if the template has user_data parameter
                                if '{user_data}' in self.qa_prompt_template:
                                    formatted_prompt = self.qa_prompt_template.format(
                                        context=context, 
                                        question=question,
                                        user_data=""  # Force empty user data for evaluation
                                    )
                                else:
                                    formatted_prompt = self.qa_prompt_template.format(
                                        context=context, 
                                        question=question
                                    )
                            else:
                                # Fallback if no template is available
                                formatted_prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
                            
                            # Determine client type and call appropriate API
                            client_type = self.client.__class__.__name__
                            
                            if client_type == "MistralAI" or "mistral" in client_type.lower():
                                # Handle Mistral client
                                from llama_index.core.llms import ChatMessage
                                
                                messages = [
                                    ChatMessage(role="user", content=formatted_prompt)
                                ]
                                
                                # Use non-streaming for evaluation
                                response = self.client.chat(
                                    messages=messages,
                                    temperature=0
                                )
                                
                                # Extract answer
                                answer_text = response.message.content
                                    
                            elif "OpenAI" in client_type or client_type == "Client":
                                # Handle OpenAI-compatible clients (including Qwen)
                                response = self.client.chat.completions.create(
                                    model=self.model,
                                    messages=[{"role": "user", "content": formatted_prompt}],
                                    temperature=0
                                )
                                
                                # Extract answer
                                answer_text = response.choices[0].message.content
                            else:
                                # Generic fallback for other client types
                                # Try common patterns
                                try:
                                    # Try llama_index style completion
                                    response = self.client.complete(formatted_prompt)
                                    answer_text = response.text if hasattr(response, 'text') else str(response)
                                except:
                                    # Fall back to using the evaluator's LLM
                                    prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
                                    response = self.llm.complete(prompt)
                                    answer_text = response.text if hasattr(response, 'text') else str(response)
                        else:
                            # Fall back to using the evaluator's LLM
                            prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
                            response = self.llm.complete(prompt)
                            answer_text = response.text if hasattr(response, 'text') else str(response)
                    except Exception as e:
                        print(f"Error generating answer: {str(e)}")
                        answer_text = f"Error generating answer: {str(e)}"
                    
                    return {
                        "answer": answer_text,
                        "contexts": context_texts
                    }
                        
                except Exception as e:
                    print(f"Error in safe_query: {str(e)}")
                    return {
                        "answer": f"Error: {str(e)}",
                        "contexts": []
                    }
            
            # Attach the safe_query method to eval_rag
            eval_rag.safe_query = types.MethodType(safe_query, eval_rag)
            
            # Make safe_query the default query method
            eval_rag.query = eval_rag.safe_query
            
            print(f"✅ Created evaluation RAG system using Excel contexts")
            return eval_rag
            
        except Exception as e:
            print(f"❌ Error creating evaluation RAG: {str(e)}")
            import traceback
            traceback.print_exc()
            return None
    
    def extract_qa_pairs(self, max_samples=None, answerable_only=True):
        """Extract question-answer pairs from the Excel dataset"""
        qa_pairs = []
        
        if self.dataset.empty:
            print("❌ No valid data found")
            return []
        
        # Check required columns
        required_columns = ['question', 'context']
        missing_cols = [col for col in required_columns if col not in self.dataset.columns]
        
        if missing_cols:
            print(f"❌ Missing required columns: {missing_cols}")
            return []
        
        # Extract QA pairs
        print("🔍 Extracting question-answer pairs...")
        
        for idx, row in tqdm(self.dataset.iterrows(), total=len(self.dataset), desc="Processing QA pairs"):
            question = str(row['question']).strip() if pd.notna(row.get('question')) else ""
            context = str(row['context']).strip() if pd.notna(row.get('context')) else ""
            
            # Skip if missing essential information
            if not question or not context:
                continue
                
            # Check if the question is impossible
            is_impossible = False
            if 'is_impossible' in self.dataset.columns:
                is_impossible = str(row['is_impossible']).upper() in ['TRUE', 'VRAI', 'YES', 'OUI', '1']
                
                # Skip unanswerable questions if specified
                if answerable_only and is_impossible:
                    continue
            
            # Get answers
            all_answers = []
            ground_truth = ""
            
            # Try to get from 'answers' column
            if 'answers' in self.dataset.columns and pd.notna(row.get('answers')):
                answers_text = row['answers']
                if answers_text != 'No answer':
                    parsed_answers = self._parse_string_list(answers_text)
                    all_answers.extend([str(a) for a in parsed_answers if a])
            
            # Try alternatives like 'answer' or 'ground_truth'
            for possible_col in ['answer', 'ground_truth']:
                if possible_col in self.dataset.columns and pd.notna(row.get(possible_col)):
                    all_answers.append(str(row[possible_col]))
            
            # For impossible questions, use empty string as the answer
            if is_impossible:
                ground_truth = ""
            elif all_answers:
                # Use the first answer as the primary ground truth
                ground_truth = all_answers[0]
            else:
                # If no answer found, skip
                if answerable_only:
                    continue
                ground_truth = ""
                
            # Create the QA pair entry
            qa_pair = {
                "id": str(idx),
                "question": question,
                "context": context,
                "ground_truth": ground_truth,
                "is_impossible": is_impossible,
                "all_answers": all_answers
            }
            
            qa_pairs.append(qa_pair)
            
            if max_samples and len(qa_pairs) >= max_samples:
                break
                
        print(f"✅ Extracted {len(qa_pairs)} QA pairs")
        
        return qa_pairs
    
    def evaluate(self, max_samples=10, answerable_only=True):
        """Evaluate RAG system using all RAGAS metrics defined in __init__"""
        # Extract QA pairs
        qa_pairs = self.extract_qa_pairs(max_samples=max_samples, answerable_only=answerable_only)
        
        if not qa_pairs:
            print("❌ No QA pairs available for evaluation")
            return {}
        
        print(f"🔍 Evaluating on {len(qa_pairs)} questions")
        
        # Initialize DataFrame
        df = pd.DataFrame({
            "question": [],
            "contexts": [],
            "answer": [],
            "ground_truth": []
        })
        
        # Track detailed responses for individual metrics
        detailed_results = []
        
        # Show the model that's being used for generating answers
        if hasattr(self.evaluation_rag, 'model'):
            print(f"🔍 EVALUATING RESPONSES FROM MODEL: {self.evaluation_rag.model}")
        elif hasattr(self.evaluation_rag, 'llm') and hasattr(self.evaluation_rag.llm, 'model'):
            print(f"🔍 EVALUATING RESPONSES FROM MODEL: {self.evaluation_rag.llm.model}")
        else:
            print(f"🔍 EVALUATING RESPONSES FROM ORIGINAL RAG SYSTEM")
        
        # Process each question
        for i, qa in enumerate(qa_pairs):
            question = qa["question"]
            ground_truth = qa["ground_truth"]
            
            print(f"\n{'='*80}")
            print(f"QUESTION {i+1}/{len(qa_pairs)}: {question}")
            print(f"{'='*80}")
            print(f"Ground Truth: {ground_truth}")
            
            # Get RAG response
            try:
                print("🔍 Getting RAG response...")
                rag_response = self.evaluation_rag.query(question)
                
                answer = rag_response.get("answer", "")
                contexts = rag_response.get("contexts", [])
                
                # Split contexts by newline if they're not already in a list
                if contexts and isinstance(contexts, str):
                    contexts = contexts.split("\n---\n")
                
                print(f"\n📝 RAG Answer: {answer}")
                
                print(f"\n📚 Retrieved Contexts ({len(contexts)} chunks):")
                for j, ctx in enumerate(contexts[:2]):
                    print(f"\nContext {j+1}:")
                    print(f"{ctx[:200]}..." if len(ctx) > 200 else ctx)
                if len(contexts) > 2:
                    print(f"\n... and {len(contexts)-2} more contexts")
                
                # Store detailed result for this question
                detailed_results.append({
                    "question_idx": i+1,
                    "question": question,
                    "ground_truth": ground_truth,
                    "answer": answer,
                    "contexts": contexts
                })
                    
            except ValueError as e:
                print(f"Error getting RAG response: {str(e)}")
                answer = "ERROR"
                contexts = []
                
                detailed_results.append({
                    "question_idx": i+1,
                    "question": question,
                    "ground_truth": ground_truth,
                    "answer": f"ERROR: {str(e)}",
                    "contexts": []
                })
            
            # Add to DataFrame
            df = pd.concat([df, pd.DataFrame({
                "question": [question],
                "answer": [answer],
                "contexts": [[ctx for ctx in contexts]],  # Make sure it's a list of strings
                "ground_truth": [ground_truth]
            })], ignore_index=True)
        
        # Convert to Dataset
        from datasets import Dataset
        eval_data = Dataset.from_dict(df)
        
        # Run RAGAS evaluation with all metrics defined in __init__
        if len(eval_data) > 0:
            print("\n📊 Computing RAGAS metrics...")
            
            try:
                # Use the pre-defined metrics
                from ragas import evaluate as ragas_evaluate
                
                # Use all metrics we defined in __init__
                result = ragas_evaluate(
                    dataset=eval_data,
                    metrics=self.metrics,  # Use metrics from the class
                    embeddings=self.embeddings,
                    llm=self.eval_llm
                )
                
                # Convert to pandas for display
                result_df = result.to_pandas()
                
                # Add the metrics from result_df to our detailed_results
                for i, row in result_df.iterrows():
                    if i < len(detailed_results):
                        # Add ALL metrics from the result dataframe to detailed results
                        for col in result_df.columns:
                            detailed_results[i][col] = row[col]
                
                # Display results
                print("\n" + "="*80)
                print("EVALUATION RESULTS")
                print("="*80)
                
                # Show metrics by directly using column names from the DataFrame
                for col in result_df.columns:
                    if col not in ['question', 'contexts', 'answer', 'ground_truth']:
                        if pd.api.types.is_numeric_dtype(result_df[col]):
                            print(f"{col}: {result_df[col].mean():.4f}")
                        else:
                            print(f"{col}: non-numeric column")
                
                # Display individual results for each question, showing ALL metrics
                print("\n" + "="*80)
                print("DETAILED RESULTS BY QUESTION")
                print("="*80)
                
                for item in detailed_results:
                    print(f"\nQuestion {item['question_idx']}: {item['question']}")
                    print(f"Ground Truth: {item['ground_truth']}")
                    
                    # Show all metrics including NaN values
                    print("\nMetrics:")
                    for col in result_df.columns:
                        if col not in ['question', 'contexts', 'answer', 'ground_truth']:
                            if col in item:
                                value = item[col]
                                # Handle different value types appropriately
                                if isinstance(value, (list, np.ndarray)):
                                    # For array-like values, display without detailed inspection
                                    print(f"  - {col}: [array data]")
                                elif pd.isna(value).any() if hasattr(value, 'any') else pd.isna(value):
                                    print(f"  - {col}: NaN")
                                elif isinstance(value, (int, float)):
                                    print(f"  - {col}: {value:.4f}")
                                else:
                                    # Truncate long string values for display
                                    if isinstance(value, str) and len(value) > 100:
                                        print(f"  - {col}: {value[:100]}...")
                                    else:
                                        print(f"  - {col}: {value}")
                            else:
                                print(f"  - {col}: Not calculated")
                
                return result_df
                    
            except Exception as e:
                print(f"❌ Error in RAGAS evaluation: {str(e)}")
                import traceback
                traceback.print_exc()
                return pd.DataFrame()
        
        return pd.DataFrame()


# Main execution
if __name__ == "__main__":
    try:
        # Debug the SustainabilityExpert class first
        print("Debugging SustainabilityExpert")
        debug_expert = SustainabilityExpert()
        print(f"DEBUG - Model being used: {debug_expert.model if hasattr(debug_expert, 'model') else 'Unknown'}")
        print(f"DEBUG - Client API available: {hasattr(debug_expert, 'client')}")
        
        # Initialize your RAG system
        original_rag = SustainabilityExpert()
        
        
        # Initialize the evaluator with the RAG system and Excel file
        evaluator = ExcelRAGEvaluator(
            original_rag_system=original_rag,
            excel_file="squad_2.0_for_eval.xlsx", 
            force_rebuild=False
        )
        
        # Run evaluation
        results = evaluator.evaluate(max_samples=10, answerable_only=True)
        
        print("\n✅ Evaluation complete!")
        
    except Exception as e:
        print(f"\n❌ ERROR in evaluation process: {str(e)}")
        import traceback
        traceback.print_exc()

Debugging SustainabilityExpert
Using model: mistral-small-latest
DEBUG - Model being used: mistral-small-latest
DEBUG - Client API available: True
Using model: mistral-small-latest
✅ Loaded dataset from squad_2.0_for_eval.xlsx
📊 Dataset shape: (11873, 9)
✅ Vector DB already exists at ./excel_eval_index_squad_2.0_for_eval
Evaluating model: mistral-small-latest
Using same similarity_top_k=10 as original RAG
✅ Created evaluation RAG system using Excel contexts
🔍 Extracting question-answer pairs...


Processing QA pairs:   0%|          | 18/11873 [00:00<00:07, 1483.43it/s]

✅ Extracted 10 QA pairs
🔍 Evaluating on 10 questions
🔍 EVALUATING RESPONSES FROM MODEL: mistral-small-latest

QUESTION 1/10: In what country is Normandy located?
Ground Truth: France
🔍 Getting RAG response...


✅ Reranked to top 4 documents

📝 RAG Answer: Based on the information provided in the context, Normandy is located in France. Here are some key points relevant to your company's understanding of the region:

- **Geographical Context**:
  - Normandy was established in the 10th century as a fiefdom through the treaty of Saint-Clair-sur-Epte between King Charles III of West Francia and Rollo, a Viking ruler.
  - The Duchy of Normandy initially included lands between the river Epte and the Atlantic coast, corresponding to the northern part of present-day Upper Normandy down to the river Seine.

- **Historical and Cultural Significance**:
  - The Normans, who gave their name to Normandy, were descendants of Norse raiders and pirates who settled in the region and gradually assimilated with the local Frankish and Roman-Gaulish populations.
  - The distinct cultural and ethnic identity of the Normans emerged in the first half of the 10th century and continued to evolve over the succeeding cent

Evaluating:   0%|          | 0/70 [00:00<?, ?it/s]


EVALUATION RESULTS
user_input: non-numeric column
retrieved_contexts: non-numeric column
response: non-numeric column
reference: non-numeric column
faithfulness: 0.7153
context_precision: 0.9250
context_recall: 0.7250
answer_relevancy: 0.7980
nv_accuracy: 0.9250
nv_context_relevance: 1.0000
nv_response_groundedness: 1.0000

DETAILED RESULTS BY QUESTION

Question 1: In what country is Normandy located?
Ground Truth: France

Metrics:
  - user_input: In what country is Normandy located?
  - retrieved_contexts: [array data]
  - response: Based on the information provided in the context, Normandy is located in France. Here are some key p...
  - reference: France
  - faithfulness: 1.0000
  - context_precision: 1.0000
  - context_recall: 1.0000
  - answer_relevancy: 0.6588
  - nv_accuracy: 1.0000
  - nv_context_relevance: 1.0000
  - nv_response_groundedness: 1.0000

Question 2: When were the Normans in Normandy?
Ground Truth: 10th and 11th centuries

Metrics:
  - user_input: When were the No